# 02 · Trích đặc trưng, huấn luyện head của các chuyên gia, xuất bundle

Thay cho các lệnh `src.models.extract`, `src.models.heads`, `src.models.bundle` trong README.

| Bước | Thời gian (RTX 4060) | Ghi chú |
|---|---|---|
| Trích embedding CLIP ViT-L/14 | ~25 phút | cần GPU + tải trọng số CLIP |
| Trích embedding DINOv2 ViT-B/14 | ~10 phút | |
| Head tuyến tính CLIP/DINOv2 | vài giây | trên embedding đã cache |
| CNN tần số (SRM) | vài phút | huấn luyện trên ảnh |
| Bundle cho từng split × biến thể head | vài phút | |

Notebook này cần dữ liệu thật (notebook 01). **Ở demo mode** các bước này được `build_demo` làm thay,
nên bạn có thể bỏ qua notebook này và chạy thẳng notebook 03/04.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

from src.core.config import load_config
from src.core.contracts import SplitManifest

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 40)
cfg = load_config()
print("repo:", ROOT)

repo: c:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics


In [2]:
import torch
MANIFEST = "data/normalized_v2/manifest.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_VARIANTS = ["clean", "aug"]
SPLITS = ["train_experts", "val_gating_fit", "val_gating_verify", "test_benchmark", "s4_support", "s4_eval"]
RUN = Path(MANIFEST).exists()
print("device:", DEVICE, "| manifest found:", RUN)

device: cuda | manifest found: True


## 1. Trích embedding (frozen backbone), cache theo sha256, chạy tiếp được nếu bị ngắt

In [3]:
from src.models.extract import main as extract_main
if RUN:
    for expert in ("clip", "dinov2"):
        extract_main(["--manifest", MANIFEST, "--expert", expert, "--device", DEVICE])

2026-11-22 14:12:34,441 | INFO    | aegis.src.models.extract | clip: 32170 records, 32170 to extract
W1122 14:12:52.204000 29392 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

2026-11-22 14:14:42,991 | INFO    | aegis.src.models.clip_expert | loaded CLIP openai/clip-vit-large-patch14 (feature_dim=768)
2026-11-22 14:15:04,671 | INFO    | aegis.src.models.extract | clip: 640/32170 extracted
2026-11-22 14:15:25,210 | INFO    | aegis.src.models.extract | clip: 1280/32170 extracted
2026-11-22 14:15:45,739 | INFO    | aegis.src.models.extract | clip: 1920/32170 extracted
2026-11-22 14:16:06,689 | INFO    | aegis.src.models.extract | clip: 2560/32170 extracted
2026-11-22 14:16:27,645 | INFO    | aegis.src.models.extract | clip: 3200/32170 extracted
2026-11-22 14:16:48,648 | INFO    | aegis.src.models.extract | clip: 3840/32170 extracted
2026-11-22 14:17:09,724 | INFO    | aegis.src.models.extract | clip: 4480/32170 extracted
2026-11-22 14:17:30,726 | INFO    | aegis.src.models.extract | clip: 5120/32170 extracted
2026-11-22 14:17:51,743 | INFO    | aegis.src.models.extract | clip: 5760/32170 extracted
2026-11-22 14:18:12,760 | INFO    | aegis.src.models.extract | c

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

2026-11-22 14:32:43,710 | INFO    | aegis.src.models.dinov2_expert | loaded DINOv2 facebook/dinov2-base (feature_dim=768)
2026-11-22 14:32:50,948 | INFO    | aegis.src.models.extract | dinov2: 640/32170 extracted
2026-11-22 14:32:57,793 | INFO    | aegis.src.models.extract | dinov2: 1280/32170 extracted
2026-11-22 14:33:04,756 | INFO    | aegis.src.models.extract | dinov2: 1920/32170 extracted
2026-11-22 14:33:11,873 | INFO    | aegis.src.models.extract | dinov2: 2560/32170 extracted
2026-11-22 14:33:18,976 | INFO    | aegis.src.models.extract | dinov2: 3200/32170 extracted
2026-11-22 14:33:26,069 | INFO    | aegis.src.models.extract | dinov2: 3840/32170 extracted
2026-11-22 14:33:33,216 | INFO    | aegis.src.models.extract | dinov2: 4480/32170 extracted
2026-11-22 14:33:40,481 | INFO    | aegis.src.models.extract | dinov2: 5120/32170 extracted
2026-11-22 14:33:47,697 | INFO    | aegis.src.models.extract | dinov2: 5760/32170 extracted
2026-11-22 14:33:54,886 | INFO    | aegis.src.model

## 2. Huấn luyện head cho từng chuyên gia

`clean` = chỉ ảnh sạch; `aug` = ảnh sạch + biến thể blur/JPEG ngẫu nhiên (train_aug).
Với chuyên gia tần số, `--select-freq` chọn đầu vào (SRM hoặc log-FFT) trên val_gating_fit.

In [5]:
from src.models.heads import main as heads_main
SELECT_FREQ = False
FREQ_INPUTS = ["srm", "log_magnitude_fft"] if SELECT_FREQ else ["srm"]
if RUN:
    for hv in HEAD_VARIANTS:
        for expert in ("clip", "dinov2"):
            heads_main(["--manifest", MANIFEST, "--expert", expert, "--variant", hv, "--device", DEVICE])
        for fi in FREQ_INPUTS:            # train the frequency CNN(s) BEFORE any selection
            heads_main(["--manifest", MANIFEST, "--expert", "freq", "--variant", hv, "--freq-input", fi,
                        "--device", DEVICE])
        if SELECT_FREQ:                   # compares the trained checkpoints; does not train anything
            heads_main(["--manifest", MANIFEST, "--expert", "freq", "--variant", hv, "--select-freq",
                        "--device", DEVICE])

2026-11-22 17:02:01,659 | INFO    | aegis.src.models.heads | trained clip[clean] on 4800 samples (4800 clean) -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\heads\clip_clean_v2.pt
2026-11-22 17:02:05,422 | INFO    | aegis.src.models.heads | trained dinov2[clean] on 4800 samples (4800 clean) -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\heads\dinov2_clean_v2.pt
2026-11-22 17:05:36,517 | INFO    | aegis.src.models.freq_expert | freq epoch 1/10 loss=0.6018
2026-11-22 17:08:54,279 | INFO    | aegis.src.models.freq_expert | freq epoch 2/10 loss=0.4676
2026-11-22 17:12:03,913 | INFO    | aegis.src.models.freq_expert | freq epoch 3/10 loss=0.3808
2026-11-22 17:15:05,790 | INFO    | aegis.src.models.freq_expert | freq epoch 4/10 loss=0.3333
2026-11-22 17:18:07,976 | INFO    | aegis.src.models.freq_expert | freq epoch 5/10 loss=0.2930
2026-11-22 17:21:11,153 | INFO    | aegis.src.models.freq_expert | freq epoch 6/1

## 3. Xuất bundle (logit, cosine distance, đặc trưng chất lượng) cho tầng gating

In [6]:
from src.models.bundle import main as bundle_main
if RUN:
    for hv in HEAD_VARIANTS:
        for split in SPLITS:
            bundle_main(["--manifest", MANIFEST, "--split", split, "--head-variant", hv, "--device", DEVICE])

2026-11-22 18:16:55,465 | INFO    | aegis.src.models.bundle | bundle train_experts: 8314 samples -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\bundles\clean\train_experts.pt
2026-11-22 18:17:22,305 | INFO    | aegis.src.models.bundle | bundle val_gating_fit: 3360 samples -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\bundles\clean\val_gating_fit.pt
2026-11-22 18:17:38,079 | INFO    | aegis.src.models.bundle | bundle val_gating_verify: 1440 samples -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\bundles\clean\val_gating_verify.pt
2026-11-22 18:18:55,799 | INFO    | aegis.src.models.bundle | bundle test_benchmark: 11200 samples -> C:\Users\Admin\OneDrive - Saigon University\Documents\aegis-ai-forensics\data\cache\bundles\clean\test_benchmark.pt
2026-11-22 18:19:09,542 | INFO    | aegis.src.models.bundle | bundle s4_support: 1600 samples -> C:\Users\Admin\OneDrive - Saig

## 4. Kiểm tra nhanh: AUROC của từng chuyên gia trên val_gating_fit

In [7]:
from sklearn.metrics import roc_auc_score
from src.core.contracts import EXPERT_NAMES
from src.learning.bundle_io import bundle_path, load_split
rows = []
for hv in HEAD_VARIANTS:
    if not bundle_path(cfg, "val_gating_fit", hv).exists():
        continue
    b = load_split(cfg, "val_gating_fit", hv)
    clean = b.clean_mask()
    for j, e in enumerate(EXPERT_NAMES):
        rows.append({"head": hv, "expert": e, "auroc_clean": roc_auc_score(b.label[clean], b.logits[clean, j]),
                     "auroc_all": roc_auc_score(b.label, b.logits[:, j])})
display(pd.DataFrame(rows) if rows else Markdown("_Chưa có bundle: chạy các bước ở trên với dữ liệu thật._"))

2026-11-22 18:23:20,091 | INFO    | aegis.src.learning.bundle_io | loaded bundle val_gating_fit.pt (n=3360)
2026-11-22 18:23:21,160 | INFO    | aegis.src.learning.bundle_io | loaded bundle val_gating_fit.pt (n=3360)


,head,expert,auroc_clean,auroc_all
0,clean,clip,0.995,0.873
1,clean,dinov2,0.911,0.837
2,clean,freq,0.977,0.662
3,aug,clip,0.990,0.930
4,aug,dinov2,0.897,0.848
5,aug,freq,0.968,0.835
